# Generation of useful files

It is the first notebook to execute

# 1) Atlas datasets

In [1]:
from pprint import pprint
from random import shuffle
import sys

sys.path.append("/Users/wang.sizh/workspace/atnt/geoloc-imc-2023/")

from scripts.utils.file_utils import load_json, dump_json
from scripts.ripe_atlas.atlas_api import get_atlas_anchors, get_atlas_probes
from default import USER_ANCHORS_FILE, USER_PROBES_FILE, USER_PROBES_AND_ANCHORS_FILE

# If you set reproducibility to True you should get the same datasets used in our work
# Set to False, if you want to run on today's data
reproducibility = True

## Retrieve atlas probes and anchors

In [2]:
# create and fill the probes.json dataset.
probes, probes_rejected, probes_geoloc_disputed = get_atlas_probes()
dump_json(probes, USER_PROBES_FILE)

# display number of selected and rejected probes.
print(
    f"selected probes: {len(probes)} ({round(len(probes) * 100 / (probes_rejected+len(probes)), 2)}%), rejected: {probes_rejected}, geoloc rejected: {probes_geoloc_disputed}"
)

KeyboardInterrupt: 

In [ ]:
# create and fill the anchors.json dataset.
anchors, anchors_rejected, anchors_geoloc_disputed = get_atlas_anchors()
dump_json(anchors, USER_ANCHORS_FILE)

# display number of selected and rejected anchors.
print(
    f"selected anchors: {len(anchors)} ({round(len(anchors) * 100 / (anchors_rejected+len(anchors)), 2)}%), rejected: {anchors_rejected}, geoloc rejected: {anchors_geoloc_disputed}"
)

selected anchors: 784 (57.14%), rejected: 588, geoloc rejected: 0


These two files will be filtered and updated step by step during the execution of this notebook

### test loading

In [3]:
probes = load_json(USER_PROBES_FILE)
anchors = load_json(USER_ANCHORS_FILE)

print(f"selected probes: {len(probes)}")
print(f"selected anchors: {len(anchors)}")

for i, probe in enumerate(probes):
    if i > 10:
        break
    pprint(probe)

selected probes: 11874
selected anchors: 924
{'address_v4': '45.138.229.91',
 'asn_v4': 206238,
 'country_code': 'NL',
 'geometry': {'coordinates': [4.9275, 52.3475], 'type': 'Point'},
 'id': 1}
{'address_v4': '77.174.76.85',
 'asn_v4': 1136,
 'country_code': 'NL',
 'geometry': {'coordinates': [4.9375, 52.3685], 'type': 'Point'},
 'id': 3}
{'address_v4': '81.56.147.159',
 'asn_v4': 29447,
 'country_code': 'IT',
 'geometry': {'coordinates': [12.4375, 41.8995], 'type': 'Point'},
 'id': 14}
{'address_v4': '77.174.30.45',
 'asn_v4': 1136,
 'country_code': 'NL',
 'geometry': {'coordinates': [5.9585, 52.0075], 'type': 'Point'},
 'id': 20}
{'address_v4': '84.82.235.83',
 'asn_v4': 1136,
 'country_code': 'NL',
 'geometry': {'coordinates': [4.8485, 52.3605], 'type': 'Point'},
 'id': 26}
{'address_v4': '24.253.79.127',
 'asn_v4': 22773,
 'country_code': 'US',
 'geometry': {'coordinates': [-115.0215, 35.9515], 'type': 'Point'},
 'id': 28}
{'address_v4': '76.82.155.9',
 'asn_v4': 20001,
 'country_

In [4]:
probes_and_anchors = probes + anchors
print(len(probes_and_anchors))
shuffle(probes_and_anchors)

dump_json(probes_and_anchors, USER_PROBES_AND_ANCHORS_FILE)

12798


# 2) Geographical datasets

In [5]:
from scripts.utils.file_utils import load_json, dump_json
from scripts.utils.helpers import distance
from default import (
    COUNTRIES_TXT_FILE,
    COUNTRIES_JSON_FILE,
    USER_ANCHORS_FILE,
    USER_PROBES_FILE,
)

## Get country dataset

In [6]:
countries = {}
with open(COUNTRIES_TXT_FILE, "r", encoding="utf8") as f:
    for i, row in enumerate(f.readlines()):

        row = [value.strip() for value in row.split(" ")]
        countries[row[0]] = {
            "latitude": row[1],
            "longitude": row[2],
            "name": row[3],
        }

for i, (country_code, geoloc) in enumerate(countries.items()):
    if i > 10:
        break
    print(f"{country_code} : {geoloc}")

# save results
dump_json(countries, COUNTRIES_JSON_FILE)

AD : {'latitude': '42.546245', 'longitude': '1.601554', 'name': 'Andorra'}
AE : {'latitude': '23.424076', 'longitude': '53.847818', 'name': 'United'}
AF : {'latitude': '33.93911', 'longitude': '67.709953', 'name': 'Afghanistan'}
AG : {'latitude': '17.060816', 'longitude': '-61.796428', 'name': 'Antigua'}
AI : {'latitude': '18.220554', 'longitude': '-63.068615', 'name': 'Anguilla'}
AL : {'latitude': '41.153332', 'longitude': '20.168331', 'name': 'Albania'}
AM : {'latitude': '40.069099', 'longitude': '45.038189', 'name': 'Armenia'}
AN : {'latitude': '12.226079', 'longitude': '-69.060087', 'name': 'Netherlands'}
AO : {'latitude': '-11.202692', 'longitude': '17.873887', 'name': 'Angola'}
AQ : {'latitude': '-75.250973', 'longitude': '-0.071389', 'name': 'Antarctica'}
AR : {'latitude': '-38.416097', 'longitude': '-63.616672', 'name': 'Argentina'}


## Eliminate default geolocated probes

All countries no matter their size have a default position. If a probe is located at this default location, it is eliminated.

In [7]:
probes = load_json(USER_PROBES_FILE)

anchors = load_json(USER_ANCHORS_FILE)

In [8]:
def country_filtering(probes: list, countries: dict) -> list:
    filtered_probes = []
    for probe in probes:

        # check if probe coordinates are close to default location
        try:
            country_geo = countries[probe["country_code"]]
        except KeyError as e:
            print(f"error country code {probe['country_code']} is unknown")
            continue

        # if the country code is unknown, remove probe from dataset
        country_lat = float(country_geo["latitude"])
        country_lon = float(country_geo["longitude"])

        probe_lat = float(probe["geometry"]["coordinates"][1])
        probe_lon = float(probe["geometry"]["coordinates"][0])

        dist = distance(country_lat, probe_lat, country_lon, probe_lon)

        if dist > 5:
            filtered_probes.append(probe)

    return filtered_probes

In [9]:
filtered_probes = country_filtering(probes, countries)
filtered_anchors = country_filtering(anchors, countries)

print(
    f"Number of Atlas probes kept: {len(filtered_probes)}, rejected: {len(probes) - len(filtered_probes)}"
)
print(
    f"Number of Atlas anchors kept: {len(filtered_anchors)}, rejected: {len(anchors) - len(filtered_anchors)}"
)

# save results
dump_json(filtered_anchors, USER_ANCHORS_FILE)
dump_json(filtered_probes, USER_PROBES_FILE)

error country code SS is unknown
error country code CW is unknown
error country code AX is unknown
error country code BL is unknown
error country code CW is unknown
Number of Atlas probes kept: 11774, rejected: 100
Number of Atlas anchors kept: 916, rejected: 8


# 3) Other various files

In [10]:
import math
import pickle
import radix
import requests
from copy import deepcopy

from clickhouse_driver import Client
from multiprocessing import Pool

from scripts.utils.file_utils import load_json, dump_json
from scripts.utils.helpers import haversine
from scripts.analysis.analysis import (
    compute_remove_wrongly_geolocated_probes,
    compute_rtts_per_dst_src,
)
from default import *

DB_HOST = "localhost"
GEO_REPLICATION_DB = "geolocation_replication"
ANCHORS_MESHED_PING_TABLE = f"anchors_meshed_pings"
PROBES_TO_ANCHORS_PING_TABLE = f"ping_10k_to_anchors"

LIMIT = 1000

## Generate ip level target list for all /24 prefixes

In [11]:
targets_per_prefix = {}

with open(ADDRESS_FILE, "r") as f:
    for i, row in enumerate(f.readlines()[1:]):
        row = row.split("\t")

        # get prefix from hex value
        prefix_hex = row[0]
        prefix = ["".join(x) for x in zip(*[iter(prefix_hex)] * 2)]
        prefix = [int(x, 16) for x in prefix]
        prefix = ".".join(str(x) for x in prefix)

        target_list = row[-1].strip("\n")
        target_list = target_list.split(",")

        # parse and save targets
        if target_list[0] != "-":
            for i, target in enumerate(target_list):
                target_list[i] = prefix.split(".")[:-1]
                target_list[i].append(str(int(target, 16)))
                target_list[i] = ".".join(target_list[i])

            try:
                targets_per_prefix[prefix].extend(target_list)
            except KeyError:
                targets_per_prefix[prefix] = target_list

In [12]:
dump_json(targets_per_prefix, USER_HITLIST_FILE)

print("target hitlist")
for i, prefix in enumerate(targets_per_prefix):
    if i > 10:
        break
    print("prefix:", prefix, "target hitlist:", targets_per_prefix[prefix])

target hitlist
prefix: 1.0.0.0 target hitlist: ['1.0.0.0', '1.0.0.1', '1.0.0.2']
prefix: 1.0.4.0 target hitlist: ['1.0.4.1', '1.0.4.4']
prefix: 1.0.5.0 target hitlist: ['1.0.5.1', '1.0.5.5']
prefix: 1.0.6.0 target hitlist: ['1.0.6.1', '1.0.6.6']
prefix: 1.0.7.0 target hitlist: ['1.0.7.1', '1.0.7.7']
prefix: 1.0.16.0 target hitlist: ['1.0.16.14', '1.0.16.9', '1.0.16.10', '1.0.16.11']
prefix: 1.0.64.0 target hitlist: ['1.0.64.25', '1.0.64.94', '1.0.64.95']
prefix: 1.0.65.0 target hitlist: ['1.0.65.6', '1.0.65.176', '1.0.65.243']
prefix: 1.0.66.0 target hitlist: ['1.0.66.10', '1.0.66.13', '1.0.66.205']
prefix: 1.0.67.0 target hitlist: ['1.0.67.15', '1.0.67.23', '1.0.67.43']
prefix: 1.0.68.0 target hitlist: ['1.0.68.21', '1.0.68.68', '1.0.68.131']


## Build pairwise matrix
WARNING : Time consumming section

This matrix represents the geographical distance between all the probes and anchors of the dataset

In [13]:
probes = load_json(USER_PROBES_FILE)
anchors = load_json(USER_ANCHORS_FILE)
anchors_ip_list = [anchor["address_v4"] for anchor in anchors]
probes.extend(anchors)

In [ ]:
vp_coordinates_per_ip = {}

for probe in probes:
    ip_v4_address = probe["address_v4"]
    long, lat = probe["geometry"]["coordinates"]
    vp_coordinates_per_ip[ip_v4_address] = lat, long


vp_distance_matrix = {}
vp_coordinates_per_ip_l = sorted(vp_coordinates_per_ip.items(), key=lambda x: x[0])

for i in range(len(vp_coordinates_per_ip_l)):
    vp_i, vp_i_coordinates = vp_coordinates_per_ip_l[i]
    if vp_i not in anchors_ip_list:
        continue
    for j in range(len(vp_coordinates_per_ip_l)):
        vp_j, vp_j_coordinates = vp_coordinates_per_ip_l[j]
        distance = haversine(vp_i_coordinates, vp_j_coordinates)
        vp_distance_matrix.setdefault(vp_i, {})[vp_j] = distance
        vp_distance_matrix.setdefault(vp_j, {})[vp_i] = distance


dump_json(vp_distance_matrix, USER_PAIRWISE_DISTANCE_FILE)

## Find wrongly geolocated probes

When looking at all the pings between probes, we sometimes notice violations of the speed of Internet, which means that the corresponding probes are wrongly geolocated.  
The following algorithm removes greedily the probe with the most violations until there is none.

In [ ]:
anchors = load_json(USER_ANCHORS_FILE)

vp_distance_matrix = load_json(USER_PAIRWISE_DISTANCE_FILE)

In [ ]:
vp_coordinates_per_ip = {}

for anchor in anchors:
    if (
        "address_v4" in anchor
        and "geometry" in anchor
        and "coordinates" in anchor["geometry"]
    ):
        ip_v4_address = anchor["address_v4"]
        if ip_v4_address is None:
            continue
        long, lat = anchor["geometry"]["coordinates"]
        vp_coordinates_per_ip[ip_v4_address] = lat, long

In [ ]:
removed_anchors = set()
filter = ""

rtt_per_srcs_dst = compute_rtts_per_dst_src(
    ANCHORS_MESHED_PING_TABLE, filter, threshold=300
)

removed_anchors = compute_remove_wrongly_geolocated_probes(
    rtt_per_srcs_dst, vp_coordinates_per_ip, vp_distance_matrix, removed_anchors
)

2024-09-02 11:43:35::INFO:root:analysis:: Violations: 124
2024-09-02 11:43:35::INFO:root:analysis:: Violations: 34
2024-09-02 11:43:35::INFO:root:analysis:: Violations: 22
2024-09-02 11:43:35::INFO:root:analysis:: Violations: 14
2024-09-02 11:43:35::INFO:root:analysis:: Violations: 10
2024-09-02 11:43:35::INFO:root:analysis:: Violations: 6
2024-09-02 11:43:35::INFO:root:analysis:: Violations: 2


In [ ]:
probes = load_json(USER_PROBES_FILE)
probes.extend(anchors)

In [ ]:
vp_coordinates_per_ip = {}

for probe in probes:
    if (
        "address_v4" in probe
        and "geometry" in probe
        and "coordinates" in probe["geometry"]
    ):
        ip_v4_address = probe["address_v4"]
        if ip_v4_address is None:
            continue
        long, lat = probe["geometry"]["coordinates"]
        vp_coordinates_per_ip[ip_v4_address] = lat, long

In [ ]:
removed_probes = set()

filter = ""
in_clause = f"".join([f",toIPv4('{p}')" for p in removed_anchors])[1:]
filter += f"AND dst not in ({in_clause}) AND src not in ({in_clause}) "

vp_coordinates_per_ip = {
    ip: vp_coordinates_per_ip[ip]
    for ip in vp_coordinates_per_ip
    if ip not in removed_anchors
}

rtt_per_srcs_dst = compute_rtts_per_dst_src(
    PROBES_TO_ANCHORS_PING_TABLE, filter, threshold=300
)

removed_probes = compute_remove_wrongly_geolocated_probes(
    rtt_per_srcs_dst, vp_coordinates_per_ip, vp_distance_matrix, removed_anchors
)

removed_probes.update(removed_anchors)

print(f"Removing {len(removed_probes)} probes")
dump_json(list(removed_probes), USER_REMOVED_PROBES_FILE)

2024-09-02 11:44:17::INFO:root:analysis:: Violations: 2730
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 1978
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 1496
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 1172
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 956
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 874
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 800
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 736
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 674
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 616
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 558
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 500
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 446
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 404
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 366
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 328
2024-09-02 11:44:17::INFO:root:analysis:: Violations: 294
2024-09-02

Removing 54 probes


## Remove bad results

Then the probes are removed of the dataset.  
We also remove anchors that haven't enough traceroute data to be analyzed.

### anchors

In [ ]:
anchors = load_json(USER_ANCHORS_FILE)

print(f"{len(anchors)} total anchors")

removed_probes = load_json(USER_REMOVED_PROBES_FILE)

9989 total anchors


In [ ]:
removed_probes = load_json(USER_REMOVED_PROBES_FILE)

In [ ]:
good_anchors = []
incorrect_geolocation_count = 0
not_enough_vps = 0
client = Client("127.0.0.1")

for anchor in anchors:
    if anchor["address_v4"] in removed_probes:
        incorrect_geolocation_count += 1
        continue

    target_ip = anchor["address_v4"]
    tmp_res_db = client.execute(
        f"select distinct src_addr from geolocation_replication.street_lvl_traceroutes where resp_addr = '{target_ip}' and dst_addr = '{target_ip}' and src_addr <> '{target_ip}'"
    )
    if len(tmp_res_db) < 100:
        not_enough_vps += 1
        continue
    good_anchors.append(anchor)

print(f"{len(good_anchors)} anchors to keep")
print(f"{incorrect_geolocation_count} anchors removed because of incorrect geolocation")
print(f"{not_enough_vps} anchors removed because the lack of traceroute data towards")

dump_json(good_anchors, USER_ANCHORS_FILE)

0 anchors to keep
20 anchors removed because of incorrect geolocation
721 anchors removed because the lack of traceroute data towards


### probes

In [ ]:
probes = load_json(USER_PROBES_FILE)

print(f"{len(probes)} total probes")

removed_probes = load_json(USER_REMOVED_PROBES_FILE)

9989 total probes


In [ ]:
good_probes = []
incorrect_geolocation_count = 0

for probe in probes:
    if probe["address_v4"] in removed_probes:
        incorrect_geolocation_count += 1
        continue
    good_probes.append(probe)

print(f"{len(good_probes)} probes to keep")
print(f"{incorrect_geolocation_count} probes removed because of incorrect geolocation")

dump_json(good_probes, USER_PROBES_FILE)

9952 probes to keep
37 probes removed because of incorrect geolocation


## Create removed and filtered probes files

All the probes removed from the beggining of the notebook are saved in the removed_probes.json file.  

All the probes that are in the clickhouse database but not in the current dataset are saved in the filtered_probes.json file.  
Later, they will be used to create a filter and collect only relevant data.

### Anchors

In [ ]:
anchors = load_json(USER_ANCHORS_FILE)

filter = ""

# clickhouse is required here
rtt_per_srcs_dst = compute_rtts_per_dst_src(
    ANCHORS_MESHED_PING_TABLE, filter, threshold=100
)

In [ ]:
for anchor in anchors:
    if anchor["address_v4"] not in rtt_per_srcs_dst:
        index = anchors.index(anchor)
        anchors.pop(index)
        print(anchor)

print(len(anchors))
dump_json(anchors, USER_ANCHORS_FILE)

anchors_list = [anchor["address_v4"] for anchor in anchors]

0


In [ ]:
# Remove probes that are in the meshed table but not in the dataset
filtered_probes = []
copy = deepcopy(rtt_per_srcs_dst)
for anchor in copy:
    if anchor not in anchors_list:
        filtered_probes.append(anchor)

copy = deepcopy(rtt_per_srcs_dst)
for anchor in copy:
    for element in copy[anchor]:
        if element not in anchors_list:
            filtered_probes.append(element)

### Probes

In [ ]:
anchors = load_json(USER_ANCHORS_FILE)
anchors_list = [anchor["address_v4"] for anchor in anchors]

probes = load_json(USER_PROBES_FILE)
probes_list = [probe["address_v4"] for probe in probes]
probes_list.extend(anchors_list)


filter = ""

# clickhouse is required here
rtt_per_srcs_dst = compute_rtts_per_dst_src(
    PROBES_TO_ANCHORS_PING_TABLE, filter, threshold=100
)

In [ ]:
# Remove probes that are in the meshed table but not in the dataset
copy = deepcopy(rtt_per_srcs_dst)
for anchor in copy:
    if anchor not in anchors_list:
        filtered_probes.append(anchor)

copy = deepcopy(rtt_per_srcs_dst)
for anchor in copy:
    for probe in copy[anchor]:
        if probe not in probes_list:
            filtered_probes.append(probe)

### Global

# CODE NOTE TESTED AFTER THIS POINT

In [ ]:
original_probes = load_json(PROBES_AND_USER_ANCHORS_FILE)
good_anchors = load_json(USER_ANCHORS_FILE)
good_probes = load_json(USER_PROBES_FILE)

final_probes = []
removed_probes = []
for probe in original_probes:
    if probe in good_anchors or probe in good_probes:
        final_probes.append(probe)
    else:
        removed_probes.append(probe["address_v4"])

dump_json(final_probes, PROBES_AND_USER_ANCHORS_FILE)
dump_json(removed_probes, REMOVED_PROBES_FILE)

NameError: name 'PROBES_AND_USER_ANCHORS_FILE' is not defined

In [ ]:
filtered_probes.extend(removed_probes)
dump_json(list(set(filtered_probes)), FILTERED_PROBES_FILE)

## Update pairwise

The pairwise matrix is updated : incorrect probes are removed.

In [ ]:
pairwise_probes = load_json(PAIRWISE_DISTANCE_FILE)
removed_probes = load_json(REMOVED_PROBES_FILE)

In [ ]:
# Remove entries with removed probes

for removed_probe in removed_probes:
    try:
        del pairwise_probes[removed_probe]
    except KeyError:
        continue

copy = deepcopy(pairwise_probes)
for probe in copy:
    for removed_probe in removed_probes:
        try:
            del pairwise_probes[probe][removed_probe]
        except KeyError:
            continue

In [ ]:
dump_json(pairwise_probes, PAIRWISE_DISTANCE_FILE)

## Select greedy probes
WARNING : Time consumming section

Greedily compute the probe with the greatest distance to other probes

In [ ]:
def greedy_selection_probes_impl(probe, distance_per_probe, selected_probes):

    distances_log = [
        math.log(distance_per_probe[p])
        for p in selected_probes
        if p in distance_per_probe and distance_per_probe[p] > 0
    ]
    total_distance = sum(distances_log)
    return probe, total_distance

In [ ]:
vp_distance_matrix = load_json(PAIRWISE_DISTANCE_FILE)

In [ ]:
print("Starting greedy algorithm")
selected_probes = []
remaining_probes = set(vp_distance_matrix.keys())
with Pool(12) as p:
    while len(remaining_probes) > 0 and len(selected_probes) < LIMIT:
        args = []
        for probe in remaining_probes:
            args.append((probe, vp_distance_matrix[probe], selected_probes))

        results = p.starmap(greedy_selection_probes_impl, args)

        furthest_probe_from_selected, _ = max(results, key=lambda x: x[1])
        selected_probes.append(furthest_probe_from_selected)
        remaining_probes.remove(furthest_probe_from_selected)

dump_json(selected_probes, GREEDY_PROBES_FILE)

Starting greedy algorithm


## Find IP info and maxmind results

Collect information from other public geolocation databases.

In [ ]:
token = "4f6c895ec9224f"

maxmind_tree_file = GEOLITE_FILE

In [ ]:
anchors = load_json(USER_ANCHORS_FILE)

removed_probes = load_json(REMOVED_PROBES_FILE)

In [ ]:
with open(maxmind_tree_file, "rb") as f:
    tree = pickle.load(f)

ip_info_geo = {}
maxmind_geo = {}


for i, anchor in enumerate(sorted(anchors, key=lambda x: x["address_v4"])):
    ip = anchor["address_v4"]

    # ip_info
    url = f"https://ipinfo.io/{ip}?token={token}"
    result = requests.get(url).json()
    ip_info_geo[ip] = result

    # maxmind_results
    node = tree.search_best(ip)
    if node is not None:
        if "city" in node.data:
            maxmind_geo[ip] = node.data["coordinates"]

dump_json(ip_info_geo, IP_INFO_GEO_FILE)
dump_json(maxmind_geo, MAXMIND_GEO_FILE)

# 4) Finish completing geographical datasets

The following file needed the final version of probes and anchors dataset

In [ ]:
import requests
import geopandas as gpd
import rasterio

from rasterio.transform import from_bounds
from pprint import pprint
from subprocess import Popen, PIPE, STDOUT

from scripts.utils.file_utils import load_json, dump_json
from default import (
    USER_ANCHORS_FILE,
    POPULATION_CITY_FILE,
    CITIES_500_FILE,
    POPULATION_DENSITY_FILE,
)

## Get population data
WARNING : Time consumming section

For each anchor we associate some geographical information like its city, the population and density of this city.

### city coordinates

In [ ]:
anchors = load_json(USER_ANCHORS_FILE)

In [ ]:
anchors_with_location = []

for anchor in anchors:
    ip = anchor["address_v4"]
    lat = anchor["geometry"]["coordinates"][1]
    lon = anchor["geometry"]["coordinates"][0]
    country_code = anchor["country_code"]
    anchors_with_location.append(
        {"target_ip": ip, "lat": lat, "lon": lon, "country_code": country_code}
    )


anchors_with_city = []
for anchor in anchors_with_location:
    url = f"http://nominatim.openstreetmap.org/reverse?format=geojson&lat={anchor['lat']}&lon={anchor['lon']}"
    r = requests.get(url)
    elem = r.json()
    if "features" not in elem or len(elem["features"]) != 1:
        continue
    info = elem["features"][0]
    if "properties" not in info or "address" not in info["properties"]:
        continue
    address = info["properties"]["address"]
    if "city" in address:
        anchor["city"] = address["city"]
    elif "village" in address:
        anchor["city"] = address["village"]
    elif "town" in address:
        anchor["city"] = address["town"]
    elif "country" in address:
        anchor["city"] = address["country"]
    else:
        pprint(info)
        break
    anchors_with_city.append(anchor)

dump_json(anchors_with_city, POPULATION_CITY_FILE)

### city population

In [ ]:
population_by_city = {}
POPULATION_THRESHOLD = 100000

with open(CITIES_500_FILE, "r", encoding="utf8") as f:
    for row in f.readlines():
        row = [value.strip() for value in row.split("\t")]
        city = row[1]
        ascii_city = row[2]
        # Iso2 code
        country = row[8]
        population = row[14]
        city_key = f"{city}_{country}"
        ascii_city_key = f"{ascii_city}_{country}"
        if population != "":
            population = int(float(population))
            if population >= POPULATION_THRESHOLD:
                population_by_city[ascii_city_key] = population
                population_by_city[city_key] = population


print(len(population_by_city) // 2)

2601


In [ ]:
anchors_with_city = load_json(POPULATION_CITY_FILE)
anchors_with_pop = []

for anchor in anchors_with_city:
    try:
        anchor["population"] = population_by_city[
            f"{anchor['city']}_{anchor['country_code']}"
        ]
        anchors_with_pop.append(anchor)
    except:
        try:
            # unix
            cmd = f"cat {CITIES_500_FILE} | grep \"{anchor['city']}\""
            # windows
            # t = "type"
            # cmd = f"{t} {CITIES_500_FILE} | findstr \"{anchor['city']}\""
            process = Popen(cmd, stdout=PIPE, stderr=STDOUT, shell=True)
            output, err = process.communicate()
            row = output.decode().split("\t")
            anchor["population"] = int(row[14])
            anchors_with_pop.append(anchor)
        except Exception as e:
            anchor["population"] = 0
            anchors_with_pop.append(anchor)

dump_json(anchors_with_pop, POPULATION_CITY_FILE)

### city density

In [ ]:
anchors_with_pop = load_json(POPULATION_CITY_FILE)

# Load the population density data
with rasterio.open(POPULATION_DENSITY_FILE) as dataset:
    population_density = dataset.read(1)

In [ ]:
anchors_with_density = []
for anchor in anchors_with_pop:
    lat, lon = anchor["lat"], anchor["lon"]
    point = gpd.GeoDataFrame(geometry=gpd.points_from_xy([lon], [lat]))

    # Convert lat-lon to pixel coordinates
    xmin, ymin, xmax, ymax = dataset.bounds
    transform = from_bounds(xmin, ymin, xmax, ymax, dataset.width, dataset.height)
    row, col = dataset.index(lon, lat)

    # Extract the population density value
    population_density_value = population_density[row, col]
    anchor["density"] = float(population_density_value)

    anchors_with_density.append(anchor)

dump_json(anchors_with_density, POPULATION_CITY_FILE)